In [1]:
import GridMLM_tokenizers
from GridMLM_tokenizers import get_chord_melody_features
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
chord_features = GridMLM_tokenizers.CHORD_FEATURES
chord_id_features = {tokenizer.vocab[k]: v for k, v in chord_features.items()}
print(chord_id_features)

{7: {'quality': 'maj', 'root': 0, 'pitch_classes': [0, 4, 7], 'features': tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}, 36: {'quality': 'maj', 'root': 1, 'pitch_classes': [1, 5, 8], 'features': tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}, 65: {'quality': 'maj', 'root': 2, 'pitch_classes': [2, 6, 9], 'features': tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}, 94: {'quality': 'maj', 'root': 3, 'pitch_classes': [3, 7, 10], 'features': tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}, 123: {'quality': 'maj', 'root': 4, 'pitch_classes': [4, 8, 11], 'features': tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}, 152: {'quality': 'maj', 'root': 5, 'pitch_classes': [0, 5, 9], 'features': tensor([[0., 0., 1., 0., 0.

In [4]:
print(chord_features['D:maj7'])
# cf0 = get_chord_pitch_features(chord_features['D:maj7'])
# print(cf0)

{'quality': 'maj7', 'root': 2, 'pitch_classes': [1, 2, 6, 9], 'features': tensor([[0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.]])}


In [5]:
train_gjt = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_train'

In [6]:
train_dataset_gjt = CSGridMLMDataset(train_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')

Loading data file.


In [7]:
d = train_dataset_gjt[0]

In [8]:
print(d.keys())
print(d['harmony_ids'])
print(len(d['harmony_ids']))
print(d['pianoroll'].shape)

dict_keys(['harmony_ids', 'attention_mask', 'pianoroll', 'time_signature', 'h_density_complexity'])
[6, 276, 276, 194, 194, 6, 339, 339, 148, 148, 6, 276, 276, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 159, 159, 6, 332, 332, 129, 129, 6, 274, 274, 71, 71, 6, 216, 216, 13, 13, 6, 159, 159, 332, 332, 6, 339, 339, 148, 148, 6, 279, 279, 279, 279, 6, 158, 158, 148, 148, 6, 270, 270, 270, 270, 6, 73, 73, 245, 245, 6, 227, 227, 226, 226, 6, 14, 14, 151, 151]
80
(80, 13)


In [11]:
# integrate chord and melody features into the dataset items
def integrate_chord_melody_features(item, chord_id_features):
    chord_features = []
    melody_features = []
    for i in range(len(item['harmony_ids'])):
        chord_id = item['harmony_ids'][i]  # assuming one chord per item
        current_chord_features = chord_id_features.get(chord_id, None)
        if current_chord_features is not None:
            chord_features.append(current_chord_features)
            melody_features.append(get_chord_melody_features(current_chord_features, item['pianoroll'][i]))
    else:
        chord_features.append(None)  # or some default value
        melody_features.append(None)  # or some default value
    item['chord_features'] = chord_features
    item['melody_features'] = melody_features
    return item

In [14]:
d1 = integrate_chord_melody_features(d, chord_id_features)

In [15]:
print(d1['chord_features'][1])
print(d1['melody_features'][1])

{'quality': 'min7', 'root': 9, 'pitch_classes': [0, 4, 7, 9], 'features': tensor([[0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 0.]])}
tensor([[1., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 1., 1.]])
